# Tiny Kitten DDPM 64

Train a very small unconditional DDPM on your own kitten/cat images, then export a browser-ready ONNX denoiser for the playground.

This is intentionally tiny. Expect cute, rough 64x64 samples, not photorealism.

## Setup

Use a local GPU runtime. If your environment does not already have PyTorch/ONNX installed, uncomment and adapt the install cell for your CUDA version.

In [ ]:
# Optional install cell. Pick the PyTorch CUDA command that matches your machine from https://pytorch.org/get-started/locally/.
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# %pip install pillow matplotlib tqdm onnx onnxruntime onnxscript

In [ ]:
from pathlib import Path
import copy
import json
import math
import random
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

try:
    import onnxruntime as ort
except Exception:
    ort = None

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'kittens'
DATA_ZIP = ROOT / 'data' / 'kittens.zip'
OUT_DIR = ROOT / 'assets' / 'models' / 'kitten-ddpm-64'
PREVIEW_DIR = OUT_DIR / 'previews'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'kitten-ddpm-64'

IMAGE_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 60
LR = 2e-4
BASE_CHANNELS = 32
TIMESTEPS = 1000
BETA_START = 1e-4
BETA_END = 2e-2
EMA_DECAY = 0.995
SAMPLE_EVERY_EPOCHS = 5
NUM_WORKERS = 2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type != 'cuda':
    print('WARNING: CUDA is not available. This notebook will run, but training on CPU will be slow.')

OUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## Dataset

Put images in `data/kittens/` or provide `data/kittens.zip`. The loader center-crops to square, resizes to 64x64, and normalizes to `[-1, 1]`.

In [ ]:
if not DATA_DIR.exists() and DATA_ZIP.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)

class ImageFolder64(Dataset):
    exts = {'.jpg', '.jpeg', '.png', '.webp'}

    def __init__(self, root, image_size=64):
        self.root = Path(root)
        self.image_size = image_size
        self.paths = sorted(p for p in self.root.rglob('*') if p.suffix.lower() in self.exts)
        self.good = []
        self.bad = []
        for path in tqdm(self.paths, desc='checking images'):
            try:
                with Image.open(path) as img:
                    img.verify()
                self.good.append(path)
            except Exception as exc:
                self.bad.append((str(path), str(exc)))
        if not self.good:
            raise RuntimeError(f'No readable images found in {self.root}. Add images or set DATA_DIR/DATA_ZIP.')

    def __len__(self):
        return len(self.good)

    def __getitem__(self, index):
        path = self.good[index]
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img).convert('RGB')
            w, h = img.size
            side = min(w, h)
            left = (w - side) // 2
            top = (h - side) // 2
            img = img.crop((left, top, left + side, top + side)).resize((self.image_size, self.image_size), Image.Resampling.LANCZOS)
        arr = np.asarray(img).astype(np.float32) / 127.5 - 1.0
        arr = np.transpose(arr, (2, 0, 1))
        return torch.from_numpy(arr)

dataset = ImageFolder64(DATA_DIR, IMAGE_SIZE)
effective_batch_size = min(BATCH_SIZE, len(dataset))
loader = DataLoader(dataset, batch_size=effective_batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=(len(dataset) >= BATCH_SIZE))
print(f'usable images: {len(dataset)}')
print(f'skipped images: {len(dataset.bad)}')
dataset.bad[:5]

In [ ]:
def show_batch(batch, title='dataset preview', n=16):
    batch = batch[:n].detach().cpu().clamp(-1, 1)
    imgs = ((batch + 1) / 2).permute(0, 2, 3, 1).numpy()
    cols = int(math.sqrt(n))
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = np.array(axes).reshape(-1)
    for ax, img in zip(axes, imgs):
        ax.imshow(img)
        ax.axis('off')
    for ax in axes[len(imgs):]:
        ax.axis('off')
    fig.suptitle(title)
    plt.show()

preview = next(iter(loader))
show_batch(preview)

## Tiny U-Net and DDPM schedule

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -(math.log(10000) / max(half - 1, 1)))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

def group_norm(channels):
    groups = 8 if channels % 8 == 0 else 4 if channels % 4 == 0 else 1
    return nn.GroupNorm(groups, channels)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.block1 = nn.Sequential(group_norm(in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.block2 = nn.Sequential(group_norm(out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.block1(x)
        h = h + self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.block2(h)
        return h + self.skip(x)

class TinyUNet(nn.Module):
    def __init__(self, base=32, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(32),
            nn.Linear(32, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.in_conv = nn.Conv2d(3, base, 3, padding=1)
        self.down1 = ResBlock(base, base, time_dim)
        self.downsample1 = nn.Conv2d(base, base * 2, 4, stride=2, padding=1)
        self.down2 = ResBlock(base * 2, base * 2, time_dim)
        self.downsample2 = nn.Conv2d(base * 2, base * 2, 4, stride=2, padding=1)
        self.mid1 = ResBlock(base * 2, base * 2, time_dim)
        self.mid2 = ResBlock(base * 2, base * 2, time_dim)
        self.upsample2 = nn.ConvTranspose2d(base * 2, base * 2, 4, stride=2, padding=1)
        self.up2 = ResBlock(base * 4, base * 2, time_dim)
        self.upsample1 = nn.ConvTranspose2d(base * 2, base, 4, stride=2, padding=1)
        self.up1 = ResBlock(base * 2, base, time_dim)
        self.out = nn.Sequential(group_norm(base), nn.SiLU(), nn.Conv2d(base, 3, 3, padding=1))

    def forward(self, x, timestep):
        t_emb = self.time_mlp(timestep.float())
        x = self.in_conv(x)
        h1 = self.down1(x, t_emb)
        x = self.downsample1(h1)
        h2 = self.down2(x, t_emb)
        x = self.downsample2(h2)
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)
        x = self.upsample2(x)
        x = self.up2(torch.cat([x, h2], dim=1), t_emb)
        x = self.upsample1(x)
        x = self.up1(torch.cat([x, h1], dim=1), t_emb)
        return self.out(x)

def make_schedule(timesteps=1000, beta_start=1e-4, beta_end=2e-2, device='cpu'):
    betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars

betas, alphas, alpha_bars = make_schedule(TIMESTEPS, BETA_START, BETA_END, device)
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

def gather(values, timesteps, x_shape):
    out = values.gather(0, timesteps)
    return out.reshape(timesteps.shape[0], *((1,) * (len(x_shape) - 1)))

def q_sample(x0, t, noise):
    return gather(sqrt_alpha_bars, t, x0.shape) * x0 + gather(sqrt_one_minus_alpha_bars, t, x0.shape) * noise

model = TinyUNet(BASE_CHANNELS).to(device)
ema_model = copy.deepcopy(model).eval().requires_grad_(False)
param_count = sum(p.numel() for p in model.parameters())
print(f'parameters: {param_count / 1e6:.2f}M')

## Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
losses = []
start_epoch = 0
ckpt_path = CHECKPOINT_DIR / 'latest.pt'

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    ema_model.load_state_dict(ckpt['ema_model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    losses = ckpt.get('losses', [])
    start_epoch = ckpt.get('epoch', 0) + 1
    print('resumed from epoch', start_epoch)

@torch.no_grad()
def update_ema(ema, src, decay=0.995):
    for ema_p, src_p in zip(ema.parameters(), src.parameters()):
        ema_p.data.mul_(decay).add_(src_p.data, alpha=1.0 - decay)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    pbar = tqdm(loader, desc=f'epoch {epoch + 1}/{EPOCHS}')
    running = []
    for x0 in pbar:
        x0 = x0.to(device, non_blocking=True)
        t = torch.randint(0, TIMESTEPS, (x0.shape[0],), device=device, dtype=torch.long)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, noise)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            pred = model(xt, t.float())
            loss = F.mse_loss(pred, noise)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        update_ema(ema_model, model, EMA_DECAY)

        value = float(loss.detach().cpu())
        running.append(value)
        losses.append(value)
        pbar.set_postfix(loss=f'{np.mean(running):.4f}')

    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'ema_model': ema_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'losses': losses,
        'config': {'image_size': IMAGE_SIZE, 'base_channels': BASE_CHANNELS, 'timesteps': TIMESTEPS},
    }, ckpt_path)

    if (epoch + 1) % SAMPLE_EVERY_EPOCHS == 0:
        print('epoch mean loss:', np.mean(running))

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.title('training loss')
plt.xlabel('step')
plt.ylabel('MSE noise prediction')
plt.show()

## Sample with DDIM

In [ ]:
@torch.no_grad()
def sample_ddim(model, n=16, steps=40, temperature=1.0):
    model.eval()
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, device=device) * temperature
    seq = torch.linspace(TIMESTEPS - 1, 0, steps, device=device).long()
    for i, t in enumerate(tqdm(seq, desc='DDIM sampling')):
        t_batch = torch.full((n,), int(t.item()), device=device, dtype=torch.float32)
        eps = model(x, t_batch)
        ab = alpha_bars[t]
        x0 = (x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)
        x0 = x0.clamp(-1, 1)
        if i == len(seq) - 1:
            x = x0
        else:
            next_t = seq[i + 1]
            ab_next = alpha_bars[next_t]
            x = torch.sqrt(ab_next) * x0 + torch.sqrt(1 - ab_next) * eps
    return x.clamp(-1, 1)

samples = sample_ddim(ema_model, n=16, steps=40)
show_batch(samples, title='EMA DDIM samples')

## Export ONNX + metadata + preview PNGs

In [ ]:
def save_tensor_grid_png(batch, path, nrow=4):
    batch = batch.detach().cpu().clamp(-1, 1)
    imgs = ((batch + 1) / 2 * 255).byte().permute(0, 2, 3, 1).numpy()
    rows = math.ceil(len(imgs) / nrow)
    canvas = Image.new('RGB', (nrow * IMAGE_SIZE, rows * IMAGE_SIZE), color=(16, 16, 16))
    for i, arr in enumerate(imgs):
        canvas.paste(Image.fromarray(arr), ((i % nrow) * IMAGE_SIZE, (i // nrow) * IMAGE_SIZE))
    canvas.save(path)

ema_model.eval().cpu()
onnx_path = OUT_DIR / 'kitten_denoiser.onnx'
dummy_x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
dummy_t = torch.tensor([500.0], dtype=torch.float32)

try:
    torch.onnx.export(
        ema_model,
        (dummy_x, dummy_t),
        str(onnx_path),
        input_names=['sample', 'timestep'],
        output_names=['epsilon'],
        opset_version=17,
        dynamo=True,
        external_data=False,
    )
except Exception as export_error:
    print('dynamo=True export failed, falling back to legacy exporter:', export_error)
    torch.onnx.export(
        ema_model,
        (dummy_x, dummy_t),
        str(onnx_path),
        input_names=['sample', 'timestep'],
        output_names=['epsilon'],
        opset_version=17,
        do_constant_folding=True,
    )

print('ONNX size MB:', onnx_path.stat().st_size / 1024 / 1024)
if onnx_path.stat().st_size > 25 * 1024 * 1024:
    print('WARNING: model is above the 25 MB target. Reduce BASE_CHANNELS and retrain before committing.')

if ort is None:
    print('Install onnxruntime to run PyTorch-vs-ONNX verification.')
else:
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    with torch.no_grad():
        torch_out = ema_model(dummy_x, dummy_t).numpy()
    ort_out = session.run(None, {'sample': dummy_x.numpy().astype(np.float32), 'timestep': dummy_t.numpy().astype(np.float32)})[0]
    max_diff = float(np.max(np.abs(torch_out - ort_out)))
    print('max PyTorch/ONNX abs diff:', max_diff)
    assert max_diff < 1e-4, max_diff

ema_model.to(device)
samples = sample_ddim(ema_model, n=16, steps=40)
save_tensor_grid_png(samples, PREVIEW_DIR / 'sample_grid.png')
for i, img in enumerate(samples[:4]):
    save_tensor_grid_png(img.unsqueeze(0), PREVIEW_DIR / f'sample_{i + 1}.png', nrow=1)

metadata = {
    'name': 'Tiny Kitten DDPM 64',
    'status': 'trained',
    'modelPath': '/assets/models/kitten-ddpm-64/kitten_denoiser.onnx',
    'imageSize': IMAGE_SIZE,
    'channels': 3,
    'normalization': {'inputRange': [-1, 1], 'outputRange': [-1, 1], 'canvasRange': [0, 255]},
    'model': {
        'architecture': 'TinyUNet',
        'predictionType': 'epsilon',
        'baseChannels': BASE_CHANNELS,
        'channelMultipliers': [1, 2, 2],
        'timeEmbeddingDim': 128,
        'inputs': {'sample': [1, 3, IMAGE_SIZE, IMAGE_SIZE], 'timestep': [1]},
        'outputs': {'epsilon': [1, 3, IMAGE_SIZE, IMAGE_SIZE]},
    },
    'trainingSchedule': {
        'type': 'linear',
        'timesteps': TIMESTEPS,
        'betaStart': BETA_START,
        'betaEnd': BETA_END,
    },
    'browserSampling': {
        'method': 'ddim',
        'defaultSteps': 40,
        'defaultTemperature': 1.0,
        'ddimTimesteps': torch.linspace(0, TIMESTEPS - 1, 40).long().cpu().tolist(),
        'alphaBars': [float(alpha_bars[i].detach().cpu()) for i in torch.linspace(0, TIMESTEPS - 1, 40).long()],
    },
    'training': {
        'epochs': EPOCHS,
        'batchSize': BATCH_SIZE,
        'learningRate': LR,
        'emaDecay': EMA_DECAY,
        'datasetSize': len(dataset),
        'finalLoss': float(np.mean(losses[-min(len(losses), len(loader)):])) if losses else None,
    },
}

with open(OUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('wrote', onnx_path)
print('wrote', OUT_DIR / 'metadata.json')
print('wrote previews to', PREVIEW_DIR)